# Job Finder — Chatbot Deployment

**ISA632 Group Project | Phase 4 of 4**

This notebook verifies the deployed Model Serving endpoint, creates the feedback storage table, and deploys the Streamlit chatbot app to Databricks Apps.

**Run order:**
1. `01_DataPreparation.ipynb` — ingest documents, build Delta table, sync Vector Search index
2. `02_AgentDevelopment.ipynb` — build, register, and deploy the RAG agent
3. `03_Evaluation.ipynb` — run MLflow evaluation with custom scorers
4. `04_ChatbotDeployment.ipynb` ← you are here

**Prerequisites:**
- `02_AgentDevelopment.ipynb` has been run and the Model Serving endpoint is active
- `chatbot_app/` directory is present at `/Workspace/Users/boopatt@miamioh.edu/Project/chatbot_app`

## Step 1 — Configure Endpoint

Set the serving endpoint name as a variable. The authentication token is fetched automatically from the current notebook context — no hardcoded credentials needed.

In [ ]:
import os
import requests
import json

# Fetch token automatically from the Databricks notebook context
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
workspace_url = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()

ENDPOINT_NAME = "agents_isa632_7474656346303369-boopatt-getstarted_job_listings"
ENDPOINT_URL = f"{workspace_url}/serving-endpoints/{ENDPOINT_NAME}/invocations"

print(f"Workspace : {workspace_url}")
print(f"Endpoint  : {ENDPOINT_URL}")

## Step 2 — Test the Model Serving Endpoint

Send a sample query to the deployed endpoint to confirm it is active and returning valid responses. If the endpoint is cold-starting it may take 30–60 seconds to respond on the first call.

In [ ]:
def score_agent(query: str) -> dict:
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }
    payload = {
        "input": [
            {"role": "user", "content": query}
        ]
    }
    response = requests.post(ENDPOINT_URL, headers=headers, json=payload, timeout=120)
    if response.status_code != 200:
        raise Exception(f"Request failed [{response.status_code}]: {response.text}")
    return response.json()

try:
    result = score_agent("Can you tell me the JPMC company data jobs available?")
    print(json.dumps(result, indent=2))
except Exception as e:
    print(f"Error: {e}\nCheck that the endpoint '{ENDPOINT_NAME}' is active in Serving.")

## Step 3 — Create Feedback Table

Create the Delta table that stores user feedback submitted through the chatbot app (👍/👎 ratings, categories, and comments). The table uses `IF NOT EXISTS` so it is safe to re-run without overwriting existing data.

## Step 3 — Deploy the Streamlit Chatbot App

Deploy the `chatbot_app/` directory to Databricks Apps. This creates a hosted Streamlit interface for the chatbot backed by the Model Serving endpoint.

- If the app already exists, the cell will deploy an update instead of creating a new one
- The app URL is printed at the end — share it with workspace users who have **Can View** permission

In [ ]:
## Step 4 — Deploy the Streamlit Chatbot App

Deploy the `chatbot_app/` directory to Databricks Apps. This creates a hosted Streamlit interface for the chatbot backed by the Model Serving endpoint.

- If the app already exists, the cell will deploy an update instead of creating a new one
- The app URL is printed at the end — share it with workspace users who have **Can View** permission

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

app_name = "job-finder-chatbot"
source_path = "/Workspace/Users/boopatt@miamioh.edu/Project/chatbot_app"

# Create the app (skip if it already exists)
try:
    app = w.apps.create_and_wait(
        name=app_name,
        description="Job Listings Chatbot powered by RAG Agent"
    )
    print(f"App created: {app.name}")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"App '{app_name}' already exists — deploying update...")
    else:
        raise e

# Deploy from chatbot_app/ directory
deployment = w.apps.deploy_and_wait(
    app_name=app_name,
    source_code_path=source_path
)

host = w.config.host.replace("https://", "").rstrip("/")
print(f"\nDeployed successfully!")
print(f"App URL: https://{host}/apps/{app_name}")